# Products Enriched ETL

## Purpose
Transform raw Bronze products into enriched, clean Silver products table with category and pricing data.

## Input → Output
* **Source:** `big_data.bronze.products`
* **Enrichment:** `big_data.bronze.aisles`, `big_data.bronze.departments`, `big_data.bronze.prices`
* **Target:** `big_data.silver.products_enriched`
* **Primary Key:** `product_id`

## Transformations
1. Load, Type Cast and Category Enrichment - Cast numeric columns, LEFT JOIN with aisles and departments, string normalization, filter NULL PKs
2. Price Enrichment and Feature Engineering - LEFT JOIN with prices (fuzzy match), cast price, filter invalid prices, derive price_band, add timestamp

## Data Quality
* **Technical:** NOT NULL (PK), UNIQUE (PK), critical columns validation
* **Business:** Enrichment coverage check (aisle, department, price), price validation (>0 or NULL), price band distribution

## Persistence
Writes to Delta table **only if all validations pass**.

### SETUP

In [0]:
%run ../UTILS/utils

In [0]:
# PySpark imports
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DecimalType

In [0]:
# Schema configuration
source_schema = "big_data.bronze"
target_schema = "big_data.silver"

# Table names
source_table = "products"
target_table = "products_enriched"

# Enrichment tables
aisles_table = "aisles"
departments_table = "departments"
prices_table = "prices"

# Primary Key columns for validation
primary_key_columns = ["product_id"]

# Critical columns (NOT NULL required)
critical_columns = ["product_name", "aisle_id", "department_id"]

# Print configuration
print("Configuration:")
print(f"  Source: {source_schema}.{source_table}")
print(f"  Target: {target_schema}.{target_table}")
print(f"  Enrichment: {aisles_table}, {departments_table}, {prices_table}")
print(f"  Primary Key: {', '.join(primary_key_columns)}")

### TRANSFORMATION

In [0]:
print("Step 1: Loading, casting types and enriching with categories...")

# Load products and apply type casting
products_df = spark.table(f"{source_schema}.{source_table}") \
    .withColumn("product_id",    F.col("product_id").cast(IntegerType())) \
    .withColumn("aisle_id",      F.col("aisle_id").cast(IntegerType())) \
    .withColumn("department_id", F.col("department_id").cast(IntegerType())) \
    .withColumn("product_name",  F.trim(F.col("product_name"))) \
    .filter(F.col("product_id").isNotNull()) \
    .filter(F.col("product_name").isNotNull())

# Join with aisles
aisles_df = spark.table(f"{source_schema}.{aisles_table}").select("aisle_id", "aisle")
products_df = products_df.join(aisles_df, on="aisle_id", how="left")

# Join with departments
departments_df = spark.table(f"{source_schema}.{departments_table}").select("department_id", "department")
products_df = products_df.join(departments_df, on="department_id", how="left")

print(f"  Loaded and enriched: {products_df.count():,} rows")
print(f"  Joined with aisles and departments")

In [0]:
print("Step 2: Enriching with prices and deriving features...")

# Normalize product names for fuzzy matching
products_df = products_df \
    .withColumn("product_name_normalized", 
        F.trim(F.regexp_replace(F.col("product_name"), "^'+|'+$", ""))
    )

# Load and prepare prices
prices_df = spark.table(f"{source_schema}.{prices_table}").select(
    F.trim(F.regexp_replace(F.col("product_name"), "^'+|'+$", "")).alias("price_product_name"),
    F.expr("try_cast(price_usd as decimal(10,2))").alias("price_usd")
)

# Join with prices (fuzzy match on normalized names)
products_silver = products_df \
    .join(
        prices_df,
        on=F.col("product_name_normalized") == F.col("price_product_name"),
        how="left"
    ) \
    .drop("price_product_name", "product_name_normalized") \
    .filter((F.col("price_usd").isNull()) | (F.col("price_usd") > 0))

# Derive price_band
products_silver = products_silver \
    .withColumn("price_band",
        F.when(F.col("price_usd").isNull(), None)
         .when(F.col("price_usd") < 2.50, "Very Low")
         .when(F.col("price_usd") < 5.00, "Low")
         .when(F.col("price_usd") < 9.00, "Medium")
         .when(F.col("price_usd") < 15.00, "High")
         .when(F.col("price_usd") < 30.00, "Premium")
         .otherwise("Luxury")
    ) \
    .withColumn("_silver_timestamp", F.current_timestamp()) \
    .drop("ingestion_timestamp", "source_file")

print(f"  Transformed: {products_silver.count():,} rows")
print(f"  Derived feature: price_band")

print("\nPreview:")
products_silver.select("product_id", "product_name", "aisle", "department", "price_usd", "price_band").show(5, truncate=False)

In [0]:
# Create final DataFrame for validation and persistence
df_result = products_silver

print(f"\nFinal DataFrame 'df_result' created: {df_result.count():,} rows")
print("\nReady for validation and persistence")

### DATA QUALITY

In [0]:
# Execute technical validations using UTILS orchestrator
validation_technical, total_rows = technical_validations(
    df=df_result,
    primary_key_columns=primary_key_columns,
    critical_columns=critical_columns,
    range_checks=[]  # No range checks needed for this table
)

print(f"\nExpected: ~49.7K rows")

In [0]:
# Business validations
print_validation_header("Business Validations")

validation_business = True

# 1. Enrichment coverage
print("\n1. Enrichment Coverage:")
with_aisle = df_result.filter(F.col("aisle").isNotNull()).count()
with_dept = df_result.filter(F.col("department").isNotNull()).count()
with_price = df_result.filter(F.col("price_usd").isNotNull()).count()

print(f"  Aisle: {with_aisle:,}/{total_rows:,} ({with_aisle/total_rows*100:.1f}%)")
print(f"  Department: {with_dept:,}/{total_rows:,} ({with_dept/total_rows*100:.1f}%)")
print(f"  Price: {with_price:,}/{total_rows:,} ({with_price/total_rows*100:.1f}%)")

if with_aisle < total_rows * 0.95 or with_dept < total_rows * 0.95:
    print("  ⚠️ WARNING: Less than 95% enrichment coverage")
    validation_business = False
else:
    print("  ✓ Enrichment coverage OK (>95%)")

# 2. Price validation (no invalid prices)
print("\n2. Price Validation:")
invalid_prices = df_result.filter((F.col("price_usd").isNotNull()) & (F.col("price_usd") <= 0)).count()
if invalid_prices == 0:
    print(f"  ✓ All prices valid (>0 or NULL)")
else:
    print(f"  ⚠️ Found {invalid_prices:,} invalid prices (<= 0)")
    validation_business = False

# 3. Price band distribution
print("\n3. Price Band Distribution:")
df_result.groupBy("price_band").count().orderBy("price_band").show()

if validation_business:
    print("\n✓ Business validations PASSED")
else:
    print("\n✗ Business validations FAILED")

In [0]:
# Combine technical and business validation results using UTILS orchestrator
validation_passed = combined_validation_result(validation_technical, validation_business)

### PERSISTENCE

In [0]:
# Conditionally persist to Delta table using UTILS function
if validation_passed:
    persist_to_delta(df_result, f"{target_schema}.{target_table}")
    print("\nNext Step: Run silver_order_products or other Silver layer notebooks")
else:
    print("\n" + "="*60)
    print("ABORTED: Validation failed - table NOT persisted")
    print("="*60)
    print("\nFix validation errors above and re-run.")